# Blocking / Candidate Generation

Blocking decides which S2/S3 records each S1 entity is ever compared with. It sets the
**recall ceiling** of the whole pipeline: a true match that is not a candidate can never be
predicted. Every extra candidate, though, costs matcher time and is one more chance of a false merge.

The code lives in `code/business_entity_resolution/src/` (`cleaning.py`, `blocking.py`) so the
matcher and the final submission package reuse it. This notebook runs it, tunes it on a train
validation sample, and writes `output/candidate_pairs.tsv` for the test set.

### What notebook 01 told us (full-data run)
| signal on true pairs | share | consequence |
|---|---|---|
| same `country` | 100% | safe **hard partition** |
| same `region` | 94.7% | too lossy as a hard partition, so used to *scope* address keys |
| same postcode | 5% | useless; not used |
| share ≥1 cleaned name token | 86.7% | name keys are the backbone, but not enough alone |
| lowest-similarity true pairs | n/a | initials (`MS` ↔ Mallick & Sons), junk names (`Novinovi`), transliterations, so address keys are needed |

### Method: multi-pass sparse TF-IDF top-K retrieval
1. Each cleaned record emits hashed **keys** from eight families:

   | family | example (from *Royal Intelligence Group, 67 Arbor Lane, Huntington, NY*) | catches |
   |---|---|---|
   | `name` | `n|royal`, `n|intelligence`, `n|royalintelligence` (joined pairs too) | name overlap, concatenations (`royalintelligence`) |
   | `phon` | `p|intlgnk` (consonant skeleton) | typos and transliteration (`inovetiv knslttents` ↔ `innovative consultants`) |
   | `join` | `j|royalintelligencegroup` | domain-style names (`reetexportsindia.com`) |
   | `nreg` | `r|ny|royal` | common name words made rare by scoping them to a region |
   | `init` | `i|rig` | initials-only names (`MS` ↔ Mallick & Sons) |
   | `addr` | `a|ny|arbor`, `a|ny|huntington`, `a|ny|67` | junk or missing names |
   | `anum` | `g|1237` (country-wide, digit-bearing tokens) | a wrong or missing region (Hyderabad listed under Andhra Pradesh) |
   | `house` | `h|67|arbor` | house number + street |

   Address numbers are canonicalised first (`0067`→`67`, `12th`/`twelfth`→`12`, `1237-1239`→`1237 1239`), and
   digit-for-letter typos in names are undone (`bi0technologies`, `5hop`). All of these were real misses in
   an early version of this notebook.
2. Keys are **IDF-weighted per country** over the S2+S3 pool; keys held by more than `cap` pool
   records (e.g. `n|trading`, `a|tx|houston`) are dropped from retrieval.
3. Each **pass** (a subset of families) retrieves the top-K S2/S3 records per S1 by cosine
   similarity, computed as a sparse matrix product and streamed over pool chunks, so memory stays
   bounded (runs on an 8 GB laptop).
4. Passes are **unioned**.

### Metrics (train validation sample, against ground truth)
* **pair recall**: share of true (S1, S2/S3) pairs that became candidates
* **ceiling F0.5**: the leaderboard metric a *perfect* matcher would score on these candidates
  (per-entity precision 1, recall = blocking recall; singletons score 1). This is the number
  blocking is optimising.
* **candidates per S1** and **reduction ratio** (share of all S1×pool pairs pruned): the matcher's workload

In [ ]:
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for base in [start, *start.parents]:
        if (base / "code/business_entity_resolution/src/blocking.py").exists():
            return base
    raise FileNotFoundError("run this notebook from inside the repository")


ROOT = find_repo_root()
SRC = str(ROOT / "code/business_entity_resolution/src")
sys.path.insert(0, SRC)
# worker processes (cleaning / key building) start fresh interpreters on Windows/macOS: pass the path on
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [SRC, os.environ.get("PYTHONPATH")]))
import blocking as B   # noqa: E402
import cleaning as C   # noqa: E402

DATA_DIR = next(p for p in ROOT.glob("**/student_resource/dataset") if "__MACOSX" not in p.parts)
CLEAN_DIR = ROOT / "cleaned"            # cleaned parquet (gitignored)
CACHE_DIR = ROOT / "blocking_cache"     # hashed keys (gitignored)
OUT_DIR = ROOT / "output"

JOBS = 2               # parallel processes for cleaning / key building (each needs ~1 GB RAM)
EVAL_N = 100_000       # S1 entities in the train validation sample
SEED = 42
RUN_TEST = True        # section 7: write output/candidate_pairs.tsv (03_run_full_pipeline sets False and does it itself)
ARTIFACT_DIR = ROOT / "artifacts"

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "axes.axisbelow": True, "legend.frameon": False})
PASS_COLORS = {"name": "#2a78d6", "addr": "#eb6834", "all": "#1baf7a"}
print(ROOT)

## 1. Clean all six source files → parquet
Streams each TSV through `cleaning.clean_records` (the notebook-01 pipeline). Files already on disk
are skipped, so re-running the notebook is cheap. About 10 minutes the first time with 2 processes.

In [ ]:
t0 = time.time()
C.clean_all(DATA_DIR, CLEAN_DIR, jobs=JOBS)
print(f"{time.time() - t0:.0f}s")
sorted(p.name for p in CLEAN_DIR.glob("*.parquet"))

## 2. Build key caches
Keys are generated once per record and cached as compressed-sparse-row arrays (`.npz` per 250k rows).
Every later experiment only re-weights these arrays; nothing is re-tokenised.

In [ ]:
FILES = {(split, src): CLEAN_DIR / f"{split}_source{src[-1]}.parquet"
         for split in ("train", "test") for src in ("S1", "S2", "S3")}

t0 = time.time()
with ProcessPoolExecutor(max_workers=JOBS) as ex:
    futures = {k: ex.submit(B.cache_keys, p, CACHE_DIR) for k, p in FILES.items()}
    chunks = {k: f.result() for k, f in futures.items()}
print(f"keys built in {time.time() - t0:.0f}s")

query = {split: chunks[(split, "S1")] for split in ("train", "test")}
pool = {split: chunks[(split, "S2")] + chunks[(split, "S3")] for split in ("train", "test")}

In [ ]:
# Keys per record by family and source (first chunk of each file)
rows = []
for (split, src), paths in chunks.items():
    kc = B.load_chunk(paths[0])
    per_fam = np.bincount(kc.fam, minlength=len(B.FAMILIES)) / kc.n
    rows.append({"split": split, "source": src, **dict(zip(B.FAMILIES, per_fam)), "total": per_fam.sum()})
pd.DataFrame(rows).set_index(["split", "source"]).round(2)

## 3. Pool statistics: document frequency per country
The IDF weights and the `cap` stop-list come from here. The long tail is what makes blocking work:
most keys are held by a handful of records, and a few (`n|services`, `a|tx|houston`) by tens of thousands.

In [ ]:
t0 = time.time()
stats = {split: B.pool_stats(pool[split]) for split in ("train", "test")}
print(f"{time.time() - t0:.0f}s")

fig, ax = plt.subplots(figsize=(8, 3.4))
for c, color in zip(stats["train"].df, ["#2a78d6", "#eb6834"]):
    d = stats["train"].df[c]
    d = d[d > 0]
    xs = np.logspace(0, np.log10(d.max()), 60)
    ax.plot(xs, [(d > x).mean() for x in xs], color=color, lw=2, label=f"{c} (pool {stats['train'].n[c]:,})")
for cap in (500, 2000, 5000):
    ax.axvline(cap, color="#8a8984", lw=1, ls=":")
ax.set(xscale="log", yscale="log", xlabel="document frequency (pool records holding the key)",
       ylabel="share of keys with df > x", title="Key document frequency (train pool); dotted: candidate caps")
ax.legend()
plt.tight_layout()

## 4. Validation set
A random sample of train S1 entities, singletons included, queried against the **full** train pool
(all 10.3M S2/S3 records) so the distractor load is realistic.

In [ ]:
gt = pd.read_csv(DATA_DIR / "train/train_ground_truth.tsv", **C.READ_KW)
gt = gt.sample(EVAL_N, random_state=SEED)

s1_ids = B.chunk_ids(query["train"])
pool_ids = B.chunk_ids(pool["train"])
eval_q = np.flatnonzero(np.isin(s1_ids, gt.source1_entity_id.to_numpy().astype("S")))

# restrict the query to the sampled rows, per chunk
q_sizes = np.r_[0, np.cumsum([B.load_chunk(p).n for p in query["train"]])]
eval_rows = {p.name: eval_q[(eval_q >= lo) & (eval_q < hi)] - lo
             for p, lo, hi in zip(query["train"], q_sizes[:-1], q_sizes[1:])}

truth = gt.assign(p_id=gt.matched_entity_ids.str.split(",")).explode("p_id").query("p_id != ''")
q_of = pd.Series(np.arange(len(s1_ids)), index=s1_ids.astype(str))
need = truth.p_id.to_numpy().astype("S")
p_pos = np.flatnonzero(np.isin(pool_ids, need))
p_of = pd.Series(p_pos, index=pool_ids[p_pos].astype(str))
truth = pd.DataFrame({"q": q_of[truth.source1_entity_id].to_numpy(), "p": p_of[truth.p_id].to_numpy()})
s1_country = np.concatenate([B.load_chunk(p).country for p in query["train"]])
truth["country"] = s1_country[truth.q]
truth["source"] = np.where(pool_ids[truth.p].astype(str) < "S3", "S2", "S3")
N_POOL = len(pool_ids)
print(f"{len(eval_q):,} S1 entities, {len(truth):,} true pairs, pool {N_POOL:,}")


def report(cand: pd.DataFrame, label: str = "") -> dict:
    return {"config": label, **B.evaluate(cand, truth[["q", "p"]], EVAL_N, N_POOL)}


def recall_at_k(cand: pd.DataFrame, ks=(1, 2, 3, 5, 8, 10, 15, 20, 30, 50)) -> pd.Series:
    """Pair recall if only the top-k of this (single-pass) candidate list were kept."""
    c = cand.sort_values(["q", "score"], ascending=[True, False])
    c["rank"] = c.groupby("q").cumcount() + 1
    hit = truth.merge(c[["q", "p", "rank"]], on=["q", "p"], how="left")["rank"]
    return pd.Series({k: (hit <= k).mean() for k in ks})

## 5. Experiments
### 5.1 Which key families pull their weight?
One pass per family group, each retrieving top-50, so recall@K curves can be read off.

In [ ]:
W_NAME, W_ADDR, W_ALL = B.W_NAME, B.W_ADDR, B.W_ALL   # family weights, shared with the pipeline

single = {
    "name": B.Pass("name", W_NAME, k=50, cap=2000),
    "addr": B.Pass("addr", W_ADDR, k=50, cap=2000),
    "all": B.Pass("all", W_ALL, k=50, cap=2000),
}
t0 = time.time()
cand50 = B.retrieve(query["train"], eval_rows, pool["train"], stats["train"], list(single.values()))
print(f"{time.time() - t0:.0f}s, {len(cand50):,} rows")

rk = pd.DataFrame({name: recall_at_k(cand50[cand50["pass"] == name]) for name in single})
rk.style.format("{:.1%}").background_gradient(cmap="Blues", axis=None)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for name in single:
    ax.plot(rk.index, rk[name], marker="o", ms=4, lw=2, color=PASS_COLORS[name], label=f"{name} pass")
ax.set(xlabel="K (candidates kept per S1 from this pass)", ylabel="pair recall", xscale="log",
       title="Recall@K by pass (train validation)")
ax.set_xticks(rk.index, rk.index)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.legend()
plt.tight_layout()

### 5.2 Unions of passes at different K
The combined `all` pass ranks best on its own, but the name-only and address-only passes find
pairs the combined score buries. Union configurations, truncating each pass's top-50 list:

In [ ]:
def truncate(cand: pd.DataFrame, k_per_pass: dict[str, int]) -> pd.DataFrame:
    c = cand[cand["pass"].isin(list(k_per_pass))].sort_values(["pass", "q", "score"], ascending=[True, True, False])
    c = c[c.groupby(["pass", "q"], observed=True).cumcount() < c["pass"].map(k_per_pass).astype(int)]
    return c


configs = {
    "all@10": {"all": 10},
    "all@20": {"all": 20},
    "all@30": {"all": 30},
    "all@10 + name@5 + addr@5": {"all": 10, "name": 5, "addr": 5},
    "all@15 + name@5 + addr@5": {"all": 15, "name": 5, "addr": 5},
    "all@20 + name@10 + addr@10": {"all": 20, "name": 10, "addr": 10},
    "all@30 + name@10 + addr@10": {"all": 30, "name": 10, "addr": 10},
}
res = pd.DataFrame([report(truncate(cand50, kp), label) for label, kp in configs.items()]).set_index("config")
res.style.format({"pair_recall": "{:.2%}", "entities_fully_covered": "{:.2%}", "ceiling_f05": "{:.4f}",
                  "cands_per_s1_mean": "{:.1f}", "cands_per_s1_p95": "{:.0f}", "reduction_ratio": "{:.7f}",
                  "n_candidates": "{:,}"})

### 5.3 Frequency cap
A higher cap lets common keys through (`n|trading`, `a|mh|mumbai`): slower, and each S1 sees more
near-duplicates. Measured on the combined pass at K=20.

In [ ]:
cap_rows = []
for cap in (500, 2000, 8000):
    t0 = time.time()
    c = B.retrieve(query["train"], eval_rows, pool["train"], stats["train"], [B.Pass("all", W_ALL, k=20, cap=cap)], log=lambda *_: None)
    cap_rows.append({**report(c, f"cap={cap}"), "seconds": time.time() - t0})
pd.DataFrame(cap_rows).set_index("config")[["pair_recall", "ceiling_f05", "cands_per_s1_mean", "seconds"]] \
    .style.format({"pair_recall": "{:.2%}", "ceiling_f05": "{:.4f}", "cands_per_s1_mean": "{:.1f}", "seconds": "{:.0f}"})

## 6. Final configuration
Chosen from 5.1–5.3 (see the summary at the end): a combined pass plus narrow name-only and
address-only passes. Defined once in `blocking.FINAL_PASSES` so the full pipeline uses the same thing.

In [ ]:
FINAL_PASSES = B.FINAL_PASSES
print(FINAL_PASSES)
final = truncate(cand50, {p.name: p.k for p in FINAL_PASSES})
final_u = B.union(final)
summary = report(final_u, "final")
pd.Series(summary)

In [ ]:
# Recall by country and by matching source
hits = truth.merge(final_u[["q", "p", "passes"]], on=["q", "p"], how="left")
hits["found"] = hits.passes.notna()
display(hits.groupby(["country", "source"]).found.mean().unstack().style.format("{:.2%}"))

# Which passes find the true pairs (exclusive contributions matter most)
hits.passes.fillna("(missed)").value_counts(normalize=True).rename("share of true pairs").to_frame() \
    .style.format("{:.2%}")

In [ ]:
# Candidate-list sizes and score separation (useful for the matcher and for any score floor)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
sizes = final_u.groupby("q").size().reindex(eval_q, fill_value=0)
axes[0].hist(sizes, bins=np.arange(0, sizes.max() + 2) - 0.5, color="#2a78d6")
axes[0].set(title="Candidates per S1 entity", xlabel="candidates", yticks=[])

is_true = final_u.merge(truth[["q", "p"]].assign(t=1), on=["q", "p"], how="left").t.eq(1).to_numpy()
for lab, m, color in [("true pair", is_true, "#2a78d6"), ("other candidate", ~is_true, "#eb6834")]:
    axes[1].hist(final_u.score[m], bins=50, range=(0, 1), density=True, histtype="step", lw=2, color=color, label=lab)
axes[1].set(title="Best retrieval score of candidates", xlabel="cosine score", yticks=[])
axes[1].legend()
plt.tight_layout()

### 6.1 What does blocking still miss?
Missed true pairs, with the cleaned fields the keys were built from. These show where
cleaning or key design could still improve.

In [ ]:
def load_rows(paths: list[Path], idx: np.ndarray, cols=("entity_id", "business_name", "name_core", "business_address", "addr_norm")) -> pd.DataFrame:
    """Fetch specific global rows (repeats allowed) from the cleaned parquet files behind `paths`' key chunks."""
    files = list(dict.fromkeys(p.name.split(".")[0] for p in paths))
    uniq = np.unique(idx)
    frames, off = [], 0
    for f in files:
        pf = pq.ParquetFile(CLEAN_DIR / f"{f}.parquet")
        n = pf.metadata.num_rows
        local = uniq[(uniq >= off) & (uniq < off + n)] - off
        if len(local):
            t = pf.read(columns=list(cols)).take(local).to_pandas()
            t.index = local + off
            frames.append(t)
        off += n
    return pd.concat(frames).loc[idx]


missed = hits[~hits.found].sample(min(15, (~hits.found).sum()), random_state=1)
L = load_rows(query["train"], missed.q.to_numpy()).reset_index(drop=True)
R = load_rows(pool["train"], missed.p.to_numpy()).reset_index(drop=True)
pd.DataFrame({"S1 name": L.business_name, "S2/S3 name": R.business_name,
              "S1 core": L.name_core, "S2/S3 core": R.name_core,
              "S1 addr": L.addr_norm.str[:45], "S2/S3 addr": R.addr_norm.str[:45]})

## 7. Test set → `output/candidate_pairs.tsv`
Full test S1 (1.7M entities) against the full test pool, including the unseen `France` partition.
IDF and caps come from the *test* pool itself (unsupervised), so no train statistics leak in.
Batched (`blocking.generate_candidates`) so memory stays bounded; also saves the candidates with
their per-pass scores to `artifacts/candidates_test/` for the matcher.

In [ ]:
if RUN_TEST:
    t0 = time.time()
    per_q_test = B.generate_candidates(query["test"], pool["test"], stats["test"], ARTIFACT_DIR / "candidates_test",
                                       passes=FINAL_PASSES, tsv_path=OUT_DIR / "candidate_pairs.tsv")
    print(f"{time.time() - t0:.0f}s", B.summarise(per_q_test, sum(stats["test"].n.values())))
    test_country = np.concatenate([B.load_chunk(p).country for p in query["test"]])
    display(pd.DataFrame({"country": test_country, "n": per_q_test.n_candidates}).groupby("country").n.describe())

In [ ]:
# Format check with the official validator (candidate file only: an empty matching file is a valid subset)
import subprocess, tempfile  # noqa: E401
if RUN_TEST:
    with tempfile.TemporaryDirectory() as tmp:
        empty = Path(tmp) / "matching_results.tsv"
        pd.DataFrame({"source1_entity_id": B.chunk_ids(query["test"]).astype(str), "matched_entity_ids": ""}) \
            .to_csv(empty, sep="\t", index=False)
        r = subprocess.run([sys.executable, str(DATA_DIR.parent / "utils/validate_submission.py"),
                            "--matching", str(empty), "--candidate", str(OUT_DIR / "candidate_pairs.tsv"),
                            "--test-dir", str(DATA_DIR / "test")], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-500:])

## 8. Summary
_Preliminary, from a 10k-entity train sample during development (re-run this notebook for the 100k numbers):_

| config | pair recall | ceiling F0.5 | candidates / S1 |
|---|---|---|---|
| all@20 | 94.5% | 0.980 | 20 |
| **all@20 + name@10 + addr@10** (final) | **95.9%** | **0.985** | 27 |
| all@50 + name@50 + addr@50 | 97.4% | 0.991 | 104 |

Remaining misses are mostly (a) S2/S3 records with no address whose name is shared by many other
businesses, which no key can separate, and (b) heavy back-transliterations (`phorcyun imphotek` ↔ *fortune infotech*).